# TBTRL FourRooms DQN

Runs the original research configuration by default, with live training logs and diagnostics immediately after each trained goal. Set `SMOKE = True` only for a short execution check.


In [ ]:
import os
from pathlib import Path
from uuid import uuid4

import torch

from tbtrl.experiments.config import load_config
from tbtrl.experiments.diagnostics import NotebookDiagnostics
from tbtrl.experiments.plots import plot_comparison, plot_run
from tbtrl.experiments.runner import run_experiment

root = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "configs" / "tbtrl_fourrooms_dqn.json").is_file()
)
config = load_config(root / "configs" / "tbtrl_fourrooms_dqn.json")
SMOKE = False
DEVICE = (
    "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
)
if SMOKE:
    config = config.smoke()
output_root = Path(os.environ.get("TBTRL_OUTPUT_ROOT", str(root / "runs")))
output = output_root / f"{config.name}-{uuid4().hex[:8]}"
print("SMOKE CHECK" if SMOKE else "FULL RESEARCH RUN")
print(f"Device: {DEVICE}; seeds: {config.seeds}; goals: {config.goals}")
print(
    f"Per-goal budget: {config.training['total_steps']:,}; warmup: {config.training['warmup_steps']:,}; batch: {config.training['batch_size']}; recovery: {config.recovery['total_steps']:,}"
)
print(
    "Early stopping retains the original threshold and patience; logs print at evaluation intervals."
)
get_ipython().run_line_magic("matplotlib", "inline")

In [ ]:
diagnostics = NotebookDiagnostics(rollout_episodes=config.eval_episodes)
records = run_experiment(config, output, device=DEVICE, on_task_end=diagnostics, verbose=True)
print(f"Saved logs, metrics, per-goal diagnostics and checkpoints to {output}")

In [ ]:
figure = plot_run(output)
figure

In [ ]:
comparison = plot_comparison(output)
comparison

The default uses the original full budgets. Early stopping can finish a goal sooner after the configured threshold and patience. Smoke runs only check execution.

Each goal produces inline diagnostics before the next goal starts and saves `diagnostics.json` and figures alongside its checkpoint. `training.log` captures the same progress shown above. Embedding drift uses fixed probes; each per-goal PCA plot uses its own basis. See `docs/notebook-parity.md` for the audited settings and documented repairs to broken legacy paths.
